# Datamine RESCAT Process Example

This notebook demonstrates how to configure and run the **`rescat`** process wrapper in `dmstudio`.

## Process Description

## RESCAT

Reserve Category Modeling.

To define reserve categories in terms of the distance of the nearest sample from the centre of a cell. All fields from the input model file are copied to the output model, and in addition 3 extra fields may be added:

* **VALUE** \- the value of the nearest sample.

* **DISTANCE** \- the distance of the nearest sample from the cell centre.

* **CATEGORY** \- a reserve category identifier.

If a field **CATEGORY** already exists in one of the input files then the **CATEGORY** field in the output **MODEL** file will be of the same type (A-alpha or N-numeric) If the field does not already exist then it will be created as an alpha field with a maximum of 4 characters.

Any combination of the 3 fields may be included controlled by the value of parameter **RESCAT**. The user is prompted to enter reserve category definitions interactively.

If RESCAT = 1,3,4 or 6 then you must define reserve categories in free format: >

* Category

* Minimum Distance

* Maximum Distance

3 data items per line separated by commas. Terminate entry with a blank line or a comma in the first character position. If parameter **ELLIPSE** is non zero then the weighting parameters are prompted:

>DIP> |  Dip of Axis 1 (degrees down from horizontal).
---|---
>AZIMUTH> |  Azimuth of Axis 1 (degrees clockwise from Y).
>AXIS 1 > |  Relative length of Axis 1.
>AXIS 2 > |  Relative length of Axis 2.
>AXIS 3 > |  Relative length of Axis 3.

* **Note** (AXIS 1 is in the Y direction, if unrotated. Distances along e.g. AXIS 2 will be multiplied by the ratio AXIS 2/AXIS 1.):

### Input Files:

* **proto** (Block Model Prototype):
  Input prototype model. This may be just fields **XC, YC, ZC, XINC, YINC, ZINC, XMORIG,
  YMORIG, ZMORIG, NX, NY, NZ, IJK**. May contain cells and sub-cells.
  Required=Yes

* **in** (Undefined):
  Input sample data (sorted on X). Must contain the fields **X , Y , Z , VALUE**.
  Required=Yes

### Output Files:

* **model** (Block Model):
  Output model.
  Required=Yes

### Fields:

* **value** (Any : IN):
  Value field. Even if the **VALUE** field is not copied to the output file a **VALUE**
  field is still required (as for any other model interpolation process).
  Default=Undefined
  Required=Yes

* **x** (Numeric : IN):
  Name of X field in **IN** sample file.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Name of Y field in **IN** sample file.
  Default=Undefined
  Required=Yes

* **z** (Numeric : IN):
  Name of Z field in **IN** sample file.
  Default=Undefined
  Required=Yes

### Parameters:

* **radius**:
  Search radius.[Maximum Value]
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **rescat**:
  Reserve category parameter controlling fields in the output model.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ellipse**:
  Ellipsoid weighting parameter. If set to 1 then the user will be prompted to enter
  distance weighting parameters.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **xsubcell**:
  No. of sub-cells/cell in X.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ysubcell**:
  No. of sub-cells/cell in Y.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **zsubcell**:
  No. of sub-cells/cell in Z. Above three parameters only used if input prototype does
  not already contain cells.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **print**:
  >=1 Display co-ordinates and interpolated values.
  Range=0,1
  Values=0,1
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('rescat')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute rescat
print("Running rescat...")
dm_cmd.rescat(
    proto_i='t_mod1',  # required
    in_i='t_assays',  # required
    model_o='t_rescat_out',  # required
    value_f='optional',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    radius_p='required_val',  # required
    # rescat_p=1,  # optional
    # ellipse_p=0,  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # print_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("rescat execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_rescat_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")